## 🔬 Análise Visual das Notas dos Alunos (`student_scores.json`)

Este notebook carrega o arquivo `student_scores.json` com a estrutura de notas aninhada por trimestre (T1, T2, T3), processa os dados para um formato tabular (DataFrame) e gera visualizações interativas para analisar a performance dos alunos.

### 1. Importação das Bibliotecas

In [ ]:
import json
import pandas as pd
import plotly.express as px

# Configuração para exibir todas as colunas do DataFrame
pd.set_option('display.max_columns', None)

### 2. Carregar e Processar os Dados

O arquivo JSON possui uma estrutura aninhada por trimestres. Vamos achatá-la para criar um DataFrame onde cada linha representa o conjunto de notas de um aluno em uma disciplina. Em seguida, calcularemos as médias de cada trimestre.

In [ ]:
# Altere o caminho se o notebook não estiver na mesma pasta que a pasta 'data'
file_path = '../data/repo/plugins/student_scores.json'

with open(file_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

students_data = raw_data.get('students_data', {})

# Lista para armazenar os dados processados
processed_data = []

# Iterar sobre cada aluno e suas disciplinas
for student_id, student_info in students_data.items():
    student_name = student_info.get('name')
    qualitative_points = sum(p.get('points', 0) for p in student_info.get('daily_qualitative_points', []))

    # Iterar sobre as disciplinas do aluno
    for subject_id, subject_details in student_info.get('subjects', {}).items():
        row = {
            'ID Aluno': student_id,
            'Nome': student_name,
            'ID Disciplina': subject_id,
            'Total Pontos Qualitativos (Geral)': qualitative_points
        }
        
        # Extrair notas de cada trimestre da estrutura aninhada
        for t in ['T1', 'T2', 'T3']:
            trim_data = subject_details.get(t, {})
            for n in ['N1', 'N2', 'N3']:
                row[f'{t}_{n}'] = trim_data.get(n, 0.0)
        
        processed_data.append(row)

# Criar o DataFrame
df = pd.DataFrame(processed_data)

# Função para calcular a média de um conjunto de notas, ignorando zeros
def calculate_mean_ignoring_zeros(row, cols):
    notes = [row[c] for c in cols if row[c] > 0]
    return sum(notes) / len(notes) if notes else 0.0

# Calcular as médias trimestrais
df['Média T1'] = df.apply(lambda row: calculate_mean_ignoring_zeros(row, ['T1_N1', 'T1_N2', 'T1_N3']), axis=1)
df['Média T2'] = df.apply(lambda row: calculate_mean_ignoring_zeros(row, ['T2_N1', 'T2_N2', 'T2_N3']), axis=1)
df['Média T3'] = df.apply(lambda row: calculate_mean_ignoring_zeros(row, ['T3_N1', 'T3_N2', 'T3_N3']), axis=1)

# Exibir as primeiras linhas para verificação
df.head()

### 3. Análise Descritiva

Vamos ver um resumo estatístico das médias trimestrais.

In [ ]:
df[['Média T1', 'Média T2', 'Média T3']].describe()

### 4. Visualizações Interativas

#### Gráfico 1: Média Geral por Trimestre em Cada Disciplina
Este gráfico nos ajuda a comparar a performance média das turmas em cada trimestre por disciplina.

In [ ]:
# Agrupar por disciplina e calcular a média
avg_scores_by_subject = df.groupby('ID Disciplina')[['Média T1', 'Média T2', 'Média T3']].mean().reset_index()

# Reformatar os dados para o Plotly (formato longo)
avg_scores_melted = avg_scores_by_subject.melt(
    id_vars='ID Disciplina', 
    value_vars=['Média T1', 'Média T2', 'Média T3'], 
    var_name='Trimestre', 
    value_name='Média'
)

# Criar o gráfico de barras
fig1 = px.bar(
    avg_scores_melted, 
    x='ID Disciplina', 
    y='Média', 
    color='Trimestre', 
    barmode='group', # Agrupa as barras lado a lado
    title='Média Geral por Trimestre em Cada Disciplina',
    labels={'ID Disciplina': 'ID da Disciplina', 'Média': 'Nota Média'},
    text_auto='.2f' # Exibe o valor em cima da barra
)

fig1.show()

#### Gráfico 2: Distribuição de Pontos Qualitativos por Aluno

Este gráfico mostra quais alunos mais acumularam pontos por participação e atividades diárias.

In [ ]:
# Pegar os pontos gerais por aluno (evitando duplicatas)
qualitative_by_student = df[['Nome', 'Total Pontos Qualitativos (Geral)']].drop_duplicates().sort_values(by='Total Pontos Qualitativos (Geral)', ascending=False)

fig2 = px.bar(
    qualitative_by_student.head(20), # Mostra os 20 melhores
    x='Nome',
    y='Total Pontos Qualitativos (Geral)',
    title='Top 20 Alunos por Pontos Qualitativos Acumulados',
    labels={'Nome': 'Aluno', 'Total Pontos Qualitativos (Geral)': 'Total de Pontos'},
    text_auto='.2f'
)

fig2.update_layout(xaxis_tickangle=-45)
fig2.show()

### 5. Análise Individual de Aluno

Vamos focar em um aluno específico para ver sua performance trimestral em cada disciplina.

In [ ]:
# Escolha o aluno pelo nome
student_name_to_analyze = "LUARA CRISTINE VISGUEIRA DE SOUSA LEMOS"

# Filtrar o DataFrame para o aluno específico
student_df = df[df['Nome'] == student_name_to_analyze]

# Reformatar os dados do aluno para o formato longo
student_melted = student_df.melt(
    id_vars=['Nome', 'ID Disciplina'],
    value_vars=['Média T1', 'Média T2', 'Média T3'],
    var_name='Trimestre',
    value_name='Nota'
)

# Criar o gráfico de barras para o aluno
fig3 = px.bar(
    student_melted,
    x='ID Disciplina',
    y='Nota',
    color='Trimestre',
    barmode='group',
    title=f'Desempenho Trimestral de {student_name_to_analyze} por Disciplina',
    labels={'ID Disciplina': 'ID da Disciplina', 'Nota': 'Média do Trimestre'},
    text_auto='.2f'
)

fig3.show()

### 6. Exportar para HTML - Opção 2: Página Única Completa

Aqui vamos criar uma página HTML única que contém todos os gráficos e tabelas formatados juntos.

In [ ]:
# Opção 1: Exportar os gráficos Plotly individualmente
fig1.write_html('analise_notas/grafico_1_medias_disciplinas.html')
fig2.write_html('analise_notas/grafico_2_pontos_qualitativos.html')
fig3.write_html('analise_notas/grafico_3_desempenho_aluno.html')

print("✅ Gráficos exportados individualmente:")
print("   📊 analise_notas/grafico_1_medias_disciplinas.html")
print("   ⭐ analise_notas/grafico_2_pontos_qualitativos.html")
print("   📈 analise_notas/grafico_3_desempenho_aluno.html")

In [ ]:
# Criar uma página HTML com gráficos e tabelas
html_content = """
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Análise de Notas - Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        body { 
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: #333;
            min-height: 100vh;
            padding: 20px;
        }
        .container { 
            max-width: 1400px; 
            margin: 0 auto; 
            background: white;
            border-radius: 10px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.2);
            padding: 30px;
        }
        h1 { 
            color: #667eea; 
            text-align: center; 
            margin-bottom: 10px;
            font-size: 2.5em;
        }
        .subtitle {
            text-align: center;
            color: #666;
            margin-bottom: 40px;
            font-size: 1.1em;
        }
        .section { 
            margin-bottom: 50px; 
            page-break-inside: avoid;
        }
        .section h2 { 
            color: #764ba2; 
            border-bottom: 3px solid #667eea;
            padding-bottom: 10px;
            margin-bottom: 20px;
            font-size: 1.8em;
        }
        .chart-container { 
            margin: 20px 0;
            border: 1px solid #eee;
            border-radius: 8px;
            padding: 15px;
            background: #f9f9f9;
        }
        table { 
            width: 100%; 
            border-collapse: collapse; 
            margin: 20px 0;
        }
        th, td { 
            border: 1px solid #ddd; 
            padding: 12px; 
            text-align: left;
        }
        th { 
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            font-weight: bold;
        }
        tr:nth-child(even) { background: #f9f9f9; }
        tr:hover { background: #f0f0f0; }
        .stats {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
            gap: 20px;
            margin: 20px 0;
        }
        .stat-box {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            padding: 20px;
            border-radius: 8px;
            box-shadow: 0 4px 15px rgba(0,0,0,0.1);
        }
        .stat-label { font-size: 0.9em; opacity: 0.9; }
        .stat-value { font-size: 2em; font-weight: bold; margin-top: 10px; }
        .footer {
            text-align: center;
            color: #999;
            margin-top: 40px;
            padding-top: 20px;
            border-top: 1px solid #eee;
            font-size: 0.9em;
        }
        @media print {
            body { background: white; }
            .container { box-shadow: none; }
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>📊 Análise de Notas dos Alunos</h1>
        <p class="subtitle">Dashboard Interativo com Gráficos e Estatísticas</p>
        
        <!-- Estatísticas Resumidas -->
        <div class="section">
            <h2>📈 Estatísticas Gerais</h2>
            <div class="stats">
                <div class="stat-box">
                    <div class="stat-label">Total de Alunos</div>
                    <div class="stat-value">""" + str(df['Nome'].nunique()) + """</div>
                </div>
                <div class="stat-box">
                    <div class="stat-label">Total de Disciplinas</div>
                    <div class="stat-value">""" + str(df['ID Disciplina'].nunique()) + """</div>
                </div>
                <div class="stat-box">
                    <div class="stat-label">Média Geral T1</div>
                    <div class="stat-value">""" + f"{df['Média T1'].mean():.2f}" + """</div>
                </div>
                <div class="stat-box">
                    <div class="stat-label">Média Geral T2</div>
                    <div class="stat-value">""" + f"{df['Média T2'].mean():.2f}" + """</div>
                </div>
                <div class="stat-box">
                    <div class="stat-label">Média Geral T3</div>
                    <div class="stat-value">""" + f"{df['Média T3'].mean():.2f}" + """</div>
                </div>
            </div>
        </div>
        
        <!-- Gráfico 1 -->
        <div class="section">
            <h2>📊 Gráfico 1: Média por Trimestre por Disciplina</h2>
            <div class="chart-container" id="chart1"></div>
        </div>
        
        <!-- Gráfico 2 -->
        <div class="section">
            <h2>⭐ Gráfico 2: Top 20 Alunos por Pontos Qualitativos</h2>
            <div class="chart-container" id="chart2"></div>
        </div>
        
        <!-- Tabela de Resumo -->
        <div class="section">
            <h2>📋 Resumo de Todas as Notas</h2>
            """ + df.head(50).to_html(classes='data-table', index=False, border=0) + """
        </div>
        
        <div class="footer">
            <p>Relatório gerado automaticamente em """ + str(pd.Timestamp.now().strftime('%d/%m/%Y às %H:%M')) + """</p>
        </div>
    </div>

    <script>
        // Preparar dados para o gráfico 1
        var chart1Data = """ + fig1.to_json() + """;
        Plotly.newPlot('chart1', chart1Data.data, chart1Data.layout);
        
        // Preparar dados para o gráfico 2
        var chart2Data = """ + fig2.to_json() + """;
        Plotly.newPlot('chart2', chart2Data.data, chart2Data.layout);
    </script>
</body>
</html>
"""

# Salvar o arquivo HTML
output_path = 'analise_notas/analise_completa.html'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"✅ Página HTML completa criada com sucesso!")
print(f"📄 Arquivo: {output_path}")
print(f"📊 A página contém:")
print(f"   ✓ Estatísticas resumidas")
print(f"   ✓ Gráficos interativos do Plotly")
print(f"   ✓ Tabelas de dados em HTML")
print(f"\n💡 Abra o arquivo no navegador para visualizar!")

### 7. Exportar Tabelas Pandas como HTML Separado

In [ ]:
# Opção 3: Exportar tabelas individuais
tabelas = {
    'Dados Completos': df,
    'Médias por Disciplina': avg_scores_by_subject,
    'Pontos Qualitativos': qualitative_by_student
}

for nome, tabela in tabelas.items():
    html_table = f"""
    <html>
    <head>
        <meta charset="UTF-8">
        <style>
            body {{ font-family: Arial, sans-serif; padding: 20px; background: #f5f5f5; }}
            h1 {{ color: #333; }}
            table {{ border-collapse: collapse; width: 100%; background: white; }}
            th, td {{ border: 1px solid #ddd; padding: 10px; text-align: left; }}
            th {{ background: #667eea; color: white; }}
            tr:nth-child(even) {{ background: #f9f9f9; }}
        </style>
    </head>
    <body>
        <h1>{nome}</h1>
        {tabela.to_html(border=0)}
    </body>
    </html>
    """
    arquivo = f"analise_notas/tabela_{nome.lower().replace(' ', '_')}.html"
    with open(arquivo, 'w', encoding='utf-8') as f:
        f.write(html_table)
    print(f"✅ {arquivo}")

print("\n💾 Todas as tabelas foram exportadas como HTML!")